In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Load the specific statcast data file
statcast_path = '/content/drive/MyDrive/CIS 5450 Final Project/statcast_data_2025_clean.csv'
statcast_df = pd.read_csv(statcast_path)

# Display the first 10 rows and info about the dataset
print(f'Successfully loaded: {statcast_path}')
display(statcast_df.head(10))
print('\nData Summary:')
statcast_df.info()

Successfully loaded: /content/drive/MyDrive/CIS 5450 Final Project/statcast_data_2025_clean.csv


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2025-03-29,102.8,-1.91,5.28,"Joyce, Ben",642136,690829,NaN,foul,...,1.0,1.31,0.55,-0.55,27.9,3.536024,14.566183,25.424326,35.165429,25.923463
1,FF,2025-03-29,102.7,-1.88,5.36,"Joyce, Ben",681460,690829,NaN,ball,...,1.0,1.12,0.62,-0.62,30.3,NaN,NaN,NaN,NaN,NaN
2,FF,2025-03-29,102.7,-1.90,5.27,"Joyce, Ben",681460,690829,NaN,foul,...,1.0,1.18,0.84,-0.84,28.3,-2.837694,25.494320,27.717819,42.741440,19.943898
3,SI,2025-03-29,102.4,-3.14,6.14,"Martinez, Justin",807713,679885,NaN,ball,...,1.0,1.83,1.33,1.33,34.5,NaN,NaN,NaN,NaN,NaN
4,FF,2025-03-30,102.3,-2.13,5.30,"Joyce, Ben",672820,690829,NaN,foul,...,1.0,1.17,1.25,1.25,27.0,-0.071433,19.641675,35.527958,43.283276,18.455490
5,FF,2025-03-29,102.3,-1.98,5.38,"Joyce, Ben",681460,690829,NaN,foul,...,1.0,1.10,0.69,-0.69,29.2,3.710849,15.924940,26.227019,38.629570,26.893958
6,FF,2025-03-29,102.2,-1.98,5.89,"Miller, Mason",641487,695243,NaN,foul,...,1.0,1.03,0.95,-0.95,32.3,-21.157514,38.232253,39.869128,35.157822,15.657434
7,FF,2025-03-30,102.2,-2.61,6.01,"Martinez, Justin",683737,679885,strikeout,called_strike,...,1.0,1.05,0.99,-0.99,31.7,NaN,NaN,NaN,NaN,NaN
8,FF,2025-03-30,102.2,-2.62,6.04,"Martinez, Justin",691718,679885,strikeout,swinging_strike,...,1.0,1.01,1.12,-1.12,34.9,-1.994127,29.637940,32.236635,38.401980,12.080867
9,SI,2025-03-30,102.0,-2.90,6.03,"Martinez, Justin",663538,679885,NaN,foul,...,1.0,1.40,1.52,1.52,32.7,NaN,NaN,NaN,NaN,NaN



Data Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348986 entries, 0 to 348985
Data columns (total 99 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   pitch_type                                348750 non-null  object 
 1   game_date                                 348986 non-null  object 
 2   release_speed                             348777 non-null  float64
 3   release_pos_x                             348776 non-null  float64
 4   release_pos_z                             348776 non-null  float64
 5   player_name                               348986 non-null  object 
 6   batter                                    348986 non-null  int64  
 7   pitcher                                   348986 non-null  int64  
 8   events                                    90709 non-null   object 
 9   description                               348986 non-null  object 
 10  zone 

# Data Wrangling

We start by filtering to regular-season pitches only, dropping rows missing core fields, and building two working frames:

- **Pitch-level frame** (`pitch_df`): every pitch during the regular season.
- **PA-level frame** (`pa_df`): one row per plate appearance, keyed by `(game_pk, at_bat_number)`, using the final pitch of each PA (which carries `events`, `woba_value`, etc.).

Bat speed is only recorded on swings, so swing-level analyses will be run on the subset where `bat_speed` is non-null.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
np.random.seed(42)

# Keep regular-season pitches only. 'R' = regular, 'S' = spring, 'F/D/L/W' = playoffs.
pitch_df = statcast_df[statcast_df["game_type"] == "R"].copy()

# Drop rows missing any of the game-state fields we rely on for leverage.
required_state = ["inning", "inning_topbot", "outs_when_up",
                  "bat_score_diff", "home_win_exp", "delta_home_win_exp"]
pitch_df = pitch_df.dropna(subset=required_state).reset_index(drop=True)

# Pre-compute flags that we will reuse downstream.
pitch_df["on_1b_flag"] = pitch_df["on_1b"].notna().astype(int)
pitch_df["on_2b_flag"] = pitch_df["on_2b"].notna().astype(int)
pitch_df["on_3b_flag"] = pitch_df["on_3b"].notna().astype(int)
pitch_df["runners_state"] = (pitch_df["on_1b_flag"].astype(str)
                             + pitch_df["on_2b_flag"].astype(str)
                             + pitch_df["on_3b_flag"].astype(str))
pitch_df["abs_score_diff"] = pitch_df["bat_score_diff"].abs()

print(f"Pitch-level rows (regular season): {len(pitch_df):,}")
print(f"Unique games: {pitch_df['game_pk'].nunique():,}")
print(f"Unique batters: {pitch_df['batter'].nunique():,}")
print(f"Unique pitchers: {pitch_df['pitcher'].nunique():,}")
print(f"Pitches with bat_speed recorded: {pitch_df['bat_speed'].notna().sum():,}")

Pitch-level rows (regular season): 348,986
Unique games: 2,428
Unique batters: 671
Unique pitchers: 798
Pitches with bat_speed recorded: 163,482


In [ ]:
# Build PA-level frame: one row per plate appearance using the LAST pitch of each PA.
# The last pitch of a PA carries `events`, `woba_value`, and `woba_denom`.
pa_df = (pitch_df
         .sort_values(["game_pk", "at_bat_number", "pitch_number"])
         .groupby(["game_pk", "at_bat_number"], as_index=False)
         .tail(1)
         .reset_index(drop=True))

# Only keep PAs that actually ended (have an `events` label).
pa_df = pa_df[pa_df["events"].notna()].reset_index(drop=True)

# Per-PA aggregates from the pitch-level frame.
per_pa_agg = (pitch_df
              .groupby(["game_pk", "at_bat_number"], as_index=False)
              .agg(n_pitches=("pitch_number", "count"),
                   max_bat_speed=("bat_speed", "max"),
                   mean_bat_speed=("bat_speed", "mean"),
                   max_release_speed=("release_speed", "max"),
                   mean_release_speed=("release_speed", "mean")))
pa_df = pa_df.merge(per_pa_agg, on=["game_pk", "at_bat_number"], how="left")

print(f"Plate appearances: {len(pa_df):,}")
print(f"PAs with woba_value: {pa_df['woba_value'].notna().sum():,}")
print(f"PAs with bat_speed on at least one pitch: {pa_df['max_bat_speed'].notna().sum():,}")
pa_df[["game_pk", "at_bat_number", "batter", "pitcher", "events",
       "woba_value", "woba_denom", "n_pitches", "max_bat_speed"]].head()

# Leverage Score Engineering

Leverage index (LI) in baseball measures how much a game state's outcome can swing win probability. FanGraphs' canonical LI is defined as the expected absolute change in win expectancy from a given base-out-score-inning state, normalized so the league average is 1.0.

We use Statcast's `delta_home_win_exp` (change in home-team win probability from each pitch) as ground truth for WE movement and build an **empirical leverage index** directly from the data:

1. Define a game state as `(inning, topbot, outs, |score_diff| clipped at 6, runners_on_base_pattern)`.
2. For each state, compute the mean absolute `delta_home_win_exp` across all pitches observed in that state.
3. Normalize by the league-wide average so the mean LI ≈ 1.0. Values > 1.5 are "high leverage", > 2.0 "very high", etc.

This lets us avoid hand-picking weights for inning/outs/runners while still capturing the same concept. We also keep the late-inning flag and simple rule-based buckets for sanity checks.

In [ ]:
# Build state key on the pitch-level frame, then compute LI per state.
pitch_df["score_diff_clip"] = pitch_df["abs_score_diff"].clip(upper=6)
# Extra innings are lumped together so we have sample size.
pitch_df["inning_bucket"] = np.where(pitch_df["inning"] >= 10, 10, pitch_df["inning"])

state_cols = ["inning_bucket", "inning_topbot", "outs_when_up",
              "score_diff_clip", "runners_state"]
pitch_df["state_key"] = pitch_df[state_cols].astype(str).agg("|".join, axis=1)

# Empirical LI: mean |delta_home_win_exp| per state, normalized by the league mean.
state_li = pitch_df.groupby("state_key")["delta_home_win_exp"].apply(lambda s: s.abs().mean())
state_n  = pitch_df.groupby("state_key").size()

# Shrink small-sample states toward the league mean to avoid noisy LI estimates.
league_mean_abs_dwe = pitch_df["delta_home_win_exp"].abs().mean()
shrink_k = 50  # pseudo-count
state_li_shrunk = (state_li * state_n + league_mean_abs_dwe * shrink_k) / (state_n + shrink_k)

# Normalize to LI scale (league avg = 1.0).
leverage_index_map = state_li_shrunk / league_mean_abs_dwe
pitch_df["leverage_index"] = pitch_df["state_key"].map(leverage_index_map)

# Rebuild state key on pa_df so LI is accessible at PA level.
pa_df["score_diff_clip"] = pa_df["bat_score_diff"].abs().clip(upper=6)
pa_df["inning_bucket"]   = np.where(pa_df["inning"] >= 10, 10, pa_df["inning"])
pa_df["state_key"]       = pa_df[state_cols].astype(str).agg("|".join, axis=1)
pa_df["leverage_index"]  = pa_df["state_key"].map(leverage_index_map)

# Bucket LI into tiers we'll use throughout the analysis.
def li_tier(li):
    if pd.isna(li):   return np.nan
    if li < 0.85:     return "low"
    if li < 1.5:      return "medium"
    if li < 2.5:      return "high"
    return "very_high"

pitch_df["leverage_tier"] = pitch_df["leverage_index"].apply(li_tier)
pa_df["leverage_tier"]    = pa_df["leverage_index"].apply(li_tier)

print("Leverage index distribution (pitch level):")
print(pitch_df["leverage_index"].describe().round(3))
print("\nLeverage tier counts (PA level):")
print(pa_df["leverage_tier"].value_counts())
print(f"\nLeague mean |delta_home_win_exp|: {league_mean_abs_dwe:.4f}")
print(f"Unique states: {pitch_df['state_key'].nunique():,}")

# Exploratory Data Analysis

We now look at:
1. How the leverage index is distributed across pitches and PAs.
2. How key performance metrics (wOBA, bat speed, pitch speed, launch speed) differ across leverage tiers.
3. Sanity checks: does LI increase in later innings / close games / runners in scoring position, as we'd expect?

In [ ]:
# Sanity check: LI should rise with inning number and with closer score margins.
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. LI distribution
axes[0].hist(pitch_df["leverage_index"], bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(1.0, color="red", ls="--", label="LI = 1 (league avg)")
axes[0].set(xlabel="Leverage Index", ylabel="Pitches", title="LI distribution (pitch level)")
axes[0].legend()

# 2. Mean LI by inning
li_by_inning = (pitch_df.groupby("inning_bucket")["leverage_index"]
                .mean().reset_index())
axes[1].plot(li_by_inning["inning_bucket"], li_by_inning["leverage_index"],
             marker="o", color="darkorange")
axes[1].axhline(1.0, color="grey", ls="--")
axes[1].set(xlabel="Inning (10 = extras)", ylabel="Mean LI", title="Mean LI by inning")

# 3. Mean LI heatmap: inning × abs(score_diff)
heat = (pitch_df.groupby(["inning_bucket", "score_diff_clip"])["leverage_index"]
        .mean().unstack())
sns.heatmap(heat, ax=axes[2], cmap="rocket_r", annot=True, fmt=".2f",
            cbar_kws={"label": "Mean LI"})
axes[2].set(xlabel="|Score diff| (clipped at 6)", ylabel="Inning",
            title="Mean LI by inning × score margin")

plt.tight_layout()
plt.show()

In [ ]:
# Summary of core performance metrics by leverage tier.
tier_order = ["low", "medium", "high", "very_high"]

swing_df   = pitch_df[pitch_df["bat_speed"].notna()].copy()
contact_df = pitch_df[pitch_df["launch_speed"].notna()].copy()

def tier_table(df, cols, label):
    out = (df.assign(leverage_tier=pd.Categorical(df["leverage_tier"],
                                                  categories=tier_order, ordered=True))
             .groupby("leverage_tier")[cols]
             .agg(["mean", "count"]))
    print(f"\n=== {label} ===")
    print(out.round(3))

tier_table(swing_df,   ["bat_speed", "swing_length"], "Swings")
tier_table(contact_df, ["launch_speed", "launch_angle"], "Batted balls")
tier_table(pitch_df,   ["release_speed", "effective_speed"], "All pitches")
tier_table(pa_df.dropna(subset=["woba_value"]),
           ["woba_value"], "Plate appearances (wOBA)")

In [ ]:
# Distributions and tier-level boxplots for the four target metrics.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

def box_by_tier(ax, df, ycol, title, ylim=None):
    data = df[df["leverage_tier"].isin(tier_order)].copy()
    data["leverage_tier"] = pd.Categorical(data["leverage_tier"],
                                           categories=tier_order, ordered=True)
    sns.boxplot(data=data, x="leverage_tier", y=ycol, ax=ax,
                showfliers=False, palette="viridis", hue="leverage_tier", legend=False)
    ax.set(title=title, xlabel="Leverage tier")
    if ylim: ax.set_ylim(ylim)

box_by_tier(axes[0, 0], swing_df,   "bat_speed",     "Bat speed by leverage tier")
box_by_tier(axes[0, 1], pitch_df,   "release_speed", "Pitch release speed by leverage tier")
box_by_tier(axes[1, 0], contact_df, "launch_speed",  "Launch speed by leverage tier")
box_by_tier(axes[1, 1],
            pa_df.dropna(subset=["woba_value"]),
            "woba_value", "wOBA value by leverage tier")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between leverage and the main target variables.
corr_rows = []
for df, col, label in [(swing_df,   "bat_speed",     "bat_speed (swings)"),
                       (pitch_df,   "release_speed", "release_speed (all pitches)"),
                       (contact_df, "launch_speed",  "launch_speed (batted balls)"),
                       (pa_df.dropna(subset=["woba_value"]), "woba_value", "wOBA (PAs)")]:
    sub = df[[col, "leverage_index"]].dropna()
    pearson  = stats.pearsonr(sub[col],  sub["leverage_index"])
    spearman = stats.spearmanr(sub[col], sub["leverage_index"])
    corr_rows.append({"metric": label, "n": len(sub),
                      "pearson_r": pearson.statistic, "pearson_p": pearson.pvalue,
                      "spearman_r": spearman.statistic, "spearman_p": spearman.pvalue})
corr_summary = pd.DataFrame(corr_rows)
print("Raw correlations with LI (no controls yet):")
display(corr_summary.round(4))

# Baseline Regression Model

A raw correlation between LI and performance is confounded by *who* pitches and bats in high-leverage spots — closers throw harder, pinch hitters are weaker, etc. Our baseline regression controls for that by including fixed effects for batter and pitcher (so we compare each player to themselves) plus pitch-type and handedness controls.

We fit two headline models:

1. **Bat speed** (swing-level): `bat_speed ~ leverage_index + batter_FE + pitcher_FE + pitch_type + platoon`
2. **wOBA** (PA-level): `woba_value ~ leverage_index + batter_FE + pitcher_FE + platoon`

We report the LI coefficient, its standard error, and an in-sample R² vs. a baseline that omits LI. If LI's coefficient is indistinguishable from zero after controls, that is evidence *against* a systematic clutch effect.

In [ ]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
from scipy import sparse

def build_design(df, target_col, include_li=True,
                 batter_col="batter", pitcher_col="pitcher",
                 extra_cat=("pitch_type", "stand", "p_throws"),
                 min_player_count=30):
    """Build a sparse design matrix with player fixed effects + categorical controls.

    Players seen fewer than `min_player_count` times are bucketed into an 'other'
    category so we don't blow up dimensionality on tiny samples.
    """
    d = df.dropna(subset=[target_col] + list(extra_cat)).copy()
    for col in [batter_col, pitcher_col]:
        counts = d[col].value_counts()
        rare = counts[counts < min_player_count].index
        d[col] = d[col].where(~d[col].isin(rare), other="other").astype(str)

    cat_cols = [batter_col, pitcher_col] + list(extra_cat)
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True, drop="first")
    X_cat = ohe.fit_transform(d[cat_cols])

    if include_li:
        X_num = sparse.csr_matrix(d[["leverage_index"]].values)
        X = sparse.hstack([X_num, X_cat]).tocsr()
        feature_names = ["leverage_index"] + list(ohe.get_feature_names_out(cat_cols))
    else:
        X = X_cat.tocsr()
        feature_names = list(ohe.get_feature_names_out(cat_cols))
    y = d[target_col].values
    return X, y, feature_names, d

def fit_and_report(df, target_col, label, alpha=1.0):
    X_full,  y, names_full,  _ = build_design(df, target_col, include_li=True)
    X_base,  _, names_base,  _ = build_design(df, target_col, include_li=False)

    model_full = Ridge(alpha=alpha).fit(X_full,  y)
    model_base = Ridge(alpha=alpha).fit(X_base,  y)

    y_full = model_full.predict(X_full)
    y_base = model_base.predict(X_base)

    # Closed-form SE via residual variance and (X^T X)^-1 on the LI column alone is
    # expensive with one-hot FEs; instead we report the partialled-out OLS estimate
    # using the Frisch-Waugh-Lovell trick on the residualised LI column.
    li_idx = names_full.index("leverage_index")
    li_col = X_full[:, li_idx].toarray().ravel()
    X_ctrl = X_full[:, [i for i in range(X_full.shape[1]) if i != li_idx]]

    # Residualise target and LI on the controls (Ridge to stabilise).
    ridge_ctrl_y  = Ridge(alpha=alpha).fit(X_ctrl, y).predict(X_ctrl)
    ridge_ctrl_li = Ridge(alpha=alpha).fit(X_ctrl, li_col).predict(X_ctrl)
    y_res  = y - ridge_ctrl_y
    li_res = li_col - ridge_ctrl_li

    beta = (li_res @ y_res) / (li_res @ li_res)
    resid = y_res - beta * li_res
    dof   = len(y_res) - 1
    sigma2 = (resid @ resid) / dof
    se = np.sqrt(sigma2 / (li_res @ li_res))
    t_stat = beta / se
    p_val  = 2 * (1 - stats.norm.cdf(abs(t_stat)))

    print(f"\n=== {label}: {target_col} ~ LI + controls ===")
    print(f"  n observations      : {len(y):,}")
    print(f"  LI coefficient (β)  : {beta: .5f}")
    print(f"  Std. error          : {se: .5f}")
    print(f"  t-stat              : {t_stat: .2f}")
    print(f"  p-value             : {p_val: .4g}")
    print(f"  R² with LI          : {r2_score(y, y_full):.4f}")
    print(f"  R² without LI       : {r2_score(y, y_base):.4f}")
    print(f"  ΔR² from adding LI  : {r2_score(y, y_full) - r2_score(y, y_base):+.5f}")
    return {"target": target_col, "beta": beta, "se": se, "p": p_val,
            "r2_with_li": r2_score(y, y_full), "r2_without_li": r2_score(y, y_base), "n": len(y)}

results = []
results.append(fit_and_report(swing_df, "bat_speed", "Swings"))
# wOBA-per-PA: only PAs with a non-null woba_value (i.e., the PA resulted in a scoring outcome).
results.append(fit_and_report(pa_df.dropna(subset=["woba_value"]),
                              "woba_value", "Plate appearances"))
# Pitcher side: does release_speed change with LI after pitcher FEs?
results.append(fit_and_report(pitch_df, "release_speed", "All pitches (pitcher view)"))

results_df = pd.DataFrame(results)
print("\nSummary of baseline regressions:")
display(results_df.round(5))

# Hypothesis Testing

We pre-registered three hypotheses in the proposal; here is our first pass at each.

- **H1**: Batters' bat speed changes under pressure. Test statistic = mean of per-batter (high-LI − low-LI) bat-speed differences. Null = leverage labels are exchangeable within a batter. Permutation test.
- **H2**: Pitchers throw harder/softer in high leverage. Same construction but per-pitcher on `release_speed`. Permutation test.
- **H3**: Clutch performance is a repeatable skill. Split each batter's high-leverage PAs into two random halves; correlation of "clutch score" (high-LI wOBA − overall wOBA) across halves. We bootstrap the correlation to build a CI and compare to a shuffled null.

All three use resampling, not parametric tests, because the per-player aggregates are not guaranteed normal and the sample sizes are heterogeneous.

In [ ]:
# Shared helper: given a pitch-level frame, tag each row "high" / "low" leverage
# using the 67th-percentile LI cutoff (roughly matching the 'high' + 'very_high' tiers).
HIGH_LI_CUTOFF = pitch_df["leverage_index"].quantile(0.67)
print(f"High-LI cutoff = {HIGH_LI_CUTOFF:.3f} (67th pct of pitch-level LI)")

def label_leverage(df):
    out = df.copy()
    out["is_high_lev"] = (out["leverage_index"] >= HIGH_LI_CUTOFF).astype(int)
    return out

def per_player_diff(df, player_col, value_col, min_each=20):
    """For each player, compute mean(value_col | high) - mean(value_col | low).
    Only keep players with at least `min_each` observations in each bucket."""
    g = df.groupby([player_col, "is_high_lev"])[value_col].agg(["mean", "size"]).unstack()
    means = g["mean"]
    sizes = g["size"]
    keep = (sizes[0] >= min_each) & (sizes[1] >= min_each)
    diffs = (means[1] - means[0])[keep]
    return diffs, sizes.loc[keep]

In [ ]:
# H1: Permutation test for bat speed in high vs low leverage (per batter).
rng = np.random.default_rng(42)
N_PERM = 2000

swing_lab = label_leverage(swing_df[swing_df["leverage_index"].notna()])
obs_diffs, obs_sizes = per_player_diff(swing_lab, "batter", "bat_speed", min_each=20)
obs_stat = obs_diffs.mean()
print(f"H1: {len(obs_diffs)} batters meet the >=20 swings in each tier cutoff.")
print(f"    Observed mean (high-LI − low-LI) bat speed: {obs_stat:+.4f} mph")

# Vectorised permutation: for each batter, shuffle their high/low labels and recompute the diff.
batter_groups = [g for _, g in swing_lab.groupby("batter")
                 if len(g) > 0 and g["is_high_lev"].nunique() == 2
                 and (g["is_high_lev"] == 0).sum() >= 20
                 and (g["is_high_lev"] == 1).sum() >= 20]

def one_perm_stat():
    diffs = []
    for g in batter_groups:
        bs = g["bat_speed"].to_numpy()
        labs = rng.permutation(g["is_high_lev"].to_numpy())
        diffs.append(bs[labs == 1].mean() - bs[labs == 0].mean())
    return np.mean(diffs)

null_stats = np.array([one_perm_stat() for _ in range(N_PERM)])
p_two_sided = (np.sum(np.abs(null_stats) >= abs(obs_stat)) + 1) / (N_PERM + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_stats, bins=40, color="lightgrey", edgecolor="white")
ax.axvline(obs_stat, color="red", lw=2, label=f"Observed = {obs_stat:+.3f}")
ax.set(xlabel="Mean (high-LI − low-LI) bat speed (mph)",
       ylabel="Permutations",
       title=f"H1 permutation null — p = {p_two_sided:.3f}  (N={N_PERM})")
ax.legend()
plt.tight_layout(); plt.show()

print(f"\nH1 result: permutation p (two-sided) = {p_two_sided:.4f}")
if p_two_sided < 0.05:
    print("    Reject H0 — bat speed does differ between leverage contexts on average.")
else:
    print("    Fail to reject H0 — no systematic league-wide bat-speed shift under pressure.")

In [ ]:
# H2: Permutation test for pitch release_speed in high vs low leverage (per pitcher).
pitch_lab = label_leverage(pitch_df[pitch_df["release_speed"].notna()
                                    & pitch_df["leverage_index"].notna()])
obs_diffs2, obs_sizes2 = per_player_diff(pitch_lab, "pitcher", "release_speed", min_each=50)
obs_stat2 = obs_diffs2.mean()
print(f"H2: {len(obs_diffs2)} pitchers meet the >=50 pitches in each tier cutoff.")
print(f"    Observed mean (high-LI − low-LI) release speed: {obs_stat2:+.4f} mph")

pitcher_groups = [g for _, g in pitch_lab.groupby("pitcher")
                  if g["is_high_lev"].nunique() == 2
                  and (g["is_high_lev"] == 0).sum() >= 50
                  and (g["is_high_lev"] == 1).sum() >= 50]

def one_perm_stat_p():
    diffs = []
    for g in pitcher_groups:
        rs   = g["release_speed"].to_numpy()
        labs = rng.permutation(g["is_high_lev"].to_numpy())
        diffs.append(rs[labs == 1].mean() - rs[labs == 0].mean())
    return np.mean(diffs)

null_stats2 = np.array([one_perm_stat_p() for _ in range(N_PERM)])
p2 = (np.sum(np.abs(null_stats2) >= abs(obs_stat2)) + 1) / (N_PERM + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_stats2, bins=40, color="lightgrey", edgecolor="white")
ax.axvline(obs_stat2, color="red", lw=2, label=f"Observed = {obs_stat2:+.3f}")
ax.set(xlabel="Mean (high-LI − low-LI) release speed (mph)", ylabel="Permutations",
       title=f"H2 permutation null — p = {p2:.3f}  (N={N_PERM})")
ax.legend()
plt.tight_layout(); plt.show()

print(f"\nH2 result: permutation p (two-sided) = {p2:.4f}")

In [ ]:
# H3: Split-half reliability of "clutch" wOBA (per batter).
# Clutch score = mean wOBA on high-LI PAs − mean wOBA on low-LI PAs.
# If clutch is a real skill, the score should be self-correlated across random halves.
pa_scored = pa_df.dropna(subset=["woba_value", "leverage_index"]).copy()
pa_scored = label_leverage(pa_scored)

MIN_HIGH = 15
MIN_LOW  = 15

def compute_clutch(df):
    g = df.groupby(["batter", "is_high_lev"])["woba_value"].agg(["mean", "size"]).unstack()
    means = g["mean"]
    sizes = g["size"]
    keep = (sizes[0] >= MIN_LOW) & (sizes[1] >= MIN_HIGH)
    return (means[1] - means[0])[keep]

# Point estimate: split each batter's PAs into two random halves and correlate clutch scores.
def split_half_corr(df, seed=0):
    local_rng = np.random.default_rng(seed)
    df = df.copy()
    df["__half"] = local_rng.integers(0, 2, size=len(df))
    s1 = compute_clutch(df[df["__half"] == 0])
    s2 = compute_clutch(df[df["__half"] == 1])
    common = s1.index.intersection(s2.index)
    if len(common) < 20:
        return np.nan, len(common)
    return stats.pearsonr(s1.loc[common], s2.loc[common]).statistic, len(common)

# Bootstrap: repeat the split-half many times to get a CI on the correlation.
N_BOOT = 500
boot_corrs = []
for i in range(N_BOOT):
    r, n = split_half_corr(pa_scored, seed=i)
    if not np.isnan(r):
        boot_corrs.append(r)
boot_corrs = np.array(boot_corrs)
r_mean = boot_corrs.mean()
ci_lo, ci_hi = np.percentile(boot_corrs, [2.5, 97.5])

# Null: shuffle is_high_lev within each batter (break the signal), recompute split-half corr.
def shuffled_corr(df, seed=0):
    local_rng = np.random.default_rng(seed + 10_000)
    df = df.copy()
    df["is_high_lev"] = (df.groupby("batter")["is_high_lev"]
                          .transform(lambda s: local_rng.permutation(s.to_numpy())))
    return split_half_corr(df, seed=seed)[0]

null_corrs = np.array([shuffled_corr(pa_scored, seed=i) for i in range(200)])
null_corrs = null_corrs[~np.isnan(null_corrs)]

p3 = (np.sum(null_corrs >= r_mean) + 1) / (len(null_corrs) + 1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(null_corrs, bins=30, color="lightgrey", edgecolor="white", label="Null (shuffled)")
ax.hist(boot_corrs, bins=30, color="steelblue", alpha=0.75, edgecolor="white",
        label="Bootstrap (observed)")
ax.axvline(r_mean, color="red", lw=2, label=f"Bootstrap mean r = {r_mean:+.3f}")
ax.axvline(ci_lo,  color="red", lw=1, ls="--", label=f"95% CI = [{ci_lo:+.3f}, {ci_hi:+.3f}]")
ax.axvline(ci_hi,  color="red", lw=1, ls="--")
ax.set(xlabel="Split-half correlation of clutch wOBA",
       ylabel="Frequency",
       title=f"H3 split-half reliability — p = {p3:.3f}")
ax.legend()
plt.tight_layout(); plt.show()

print(f"H3 result: bootstrap split-half r = {r_mean:+.3f}  (95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}])")
print(f"           one-sided p vs shuffled null = {p3:.4f}")
if ci_lo > 0 and p3 < 0.05:
    print("    Evidence that clutch wOBA has some repeatable signal.")
else:
    print("    Cannot distinguish clutch wOBA from noise — consistent with the classical view.")

# Next steps for the final report

Everything above is the *baseline*. Before the final write-up we still need to:

- **Sensitivity check the leverage score**: redo the regressions with an alternative, rule-based LI (inning × outs × runners × closeness) and verify the sign/magnitude of the LI coefficient is stable.
- **Robust standard errors**: cluster SEs by batter and pitcher for the regression models (the Ridge + FWL setup here assumes iid residuals).
- **Multiple testing**: adjust p-values across the three hypotheses (Holm or BH) before drawing conclusions.
- **Classification model** (proposal): build a logistic regression / gradient-boosted model for "clutch success" (above-average wOBA in high-LI PAs) and look at feature importance.
- **Confounding controls**: our FE regression already partly handles "closers face high LI," but we should also condition on pitcher role (reliever vs. starter) and inning late.

Any nominally-significant result needs to survive (a) controls, (b) multiple testing, and (c) a split-half reliability check before we call it a real clutch effect.